# BGC-Argo Data Processing

- After getting .nc files from 0.2_*_Argopy, need to process and QC the BGC-Argo data


In [2]:
# os tools
import sys
import os
import os.path
import requests
import time
import urllib3
import shutil

# data tools
import numpy                 as np
import pandas                as pd
import xarray                as xr
from   datetime              import date, datetime, timedelta                 # for saving figures with today's date
import datetime
import scipy
from   scipy.stats           import kruskal              # for boxenplot stats
from   scipy.stats           import mannwhitneyu         # for split violin plot stats

# for all plots
import matplotlib
import matplotlib.pyplot     as plt                      # needed to make map setup
import matplotlib.colors     as colors
from   matplotlib.ticker     import EngFormatter         # for degree symbol in axis
from   cmocean               import cm as cmo
import seaborn               as     sns

# for map
import shapefile
import cartopy                                           # to make map
import matplotlib.path       as     mpath                # to draw circle for map
import cartopy.crs           as     ccrs                 # for map projection
import cartopy.feature       as     cfeature             # to add land features to map
# from   xhistogram.xarray     import histogram            # for map histogram
# from   mycolorpy             import colorlist as mcp     # to get n colors list
import pyproj  
import geopandas             as     gpd                  # for adding shapefiles of frontal zones 
from   osgeo                 import gdal
# import scikit_posthocs       as     sp                   # for stats


In [3]:
from importlib import reload
import mod_plotting as crplot

from mod_plotting import setup_SO_axes
plt.rcParams.update(crplot.my_params(size=12))

In [4]:
xr.set_options(display_expand_attrs = False)

In [5]:
import shapefile
so_fronts = shapefile.Reader('./shapefiles/fronts/so_fronts.shp') 
stf_mod   = shapefile.Reader('./shapefiles/fronts/stf_mod/stf_mod.shp')

stf  = stf_mod.shape(0).points
saf  = so_fronts.shape(1).points
pf   = so_fronts.shape(2).points
sacc = so_fronts.shape(3).points
sie  = so_fronts.shape(4).points

max_latitude:          float = -30
add_gridlines:         bool  = True
color_land:            bool  = False
land_edgecolor:        str   = 'grey'
land_facecolor:        str   = 'grey'
fontsize:              float = 10
map_facecolor:         str   = 'white'
coast_linewidth:       float = 0.3
gridlines_linewidth:   float = 0.5
girdlines_color:       str   = 'grey'
gridlines_alpha:       float = 0.5
longitude_label_color: str   = 'grey'
latitude_label_color:  str   = 'grey'

## Import .nc files from 0.2_DataProcessing_Argopy

In [6]:
# Example BGC float with pH data
example_bgc = xr.open_dataset('../data/argopy_5906030_bgc_profiles_2025Jan29.nc')

In [19]:
example_bgc.to_dataframe().reset_index().PSAL_QC.unique()

array([    1,     8,     4, 99999])

In [65]:
example_bgc

<xarray.Dataset>
Dimensions:                          (N_PROF: 166, N_LEVELS: 559)
Coordinates:
  * N_PROF                           (N_PROF) int64 0 1 2 3 ... 162 163 164 165
  * N_LEVELS                         (N_LEVELS) int64 0 1 2 3 ... 556 557 558
    LATITUDE                         (N_PROF) float64 -51.35 -51.07 ... -47.72
    LONGITUDE                        (N_PROF) float64 30.21 32.83 ... 111.0
    TIME                             (N_PROF) datetime64[ns] 2019-05-01T08:58...
Data variables: (12/54)
    BBP700                           (N_PROF, N_LEVELS) float32 nan nan ... nan
    BBP700_ADJUSTED                  (N_PROF, N_LEVELS) float32 nan nan ... nan
    BBP700_ADJUSTED_ERROR            (N_PROF) float32 nan nan nan ... nan nan
    BBP700_ADJUSTED_QC               (N_PROF, N_LEVELS) int64 0 0 ... 99999
    BBP700_DATA_MODE                 (N_PROF) object 'A' 'A' 'A' ... 'A' 'A' 'A'
    BBP700_QC                        (N_PROF, N_LEVELS) int64 0 0 ... 99999
    ...                               ...
    TEMP_ADJUSTED                    (N_PROF, N_LEVELS) float32 3.634 ... nan
    TEMP_ADJUSTED_ERROR              (N_PROF, N_LEVELS) float32 0.002 ... nan
    TEMP_ADJUSTED_QC                 (N_PROF, N_LEVELS) int64 1 1 ... 99999
    TEMP_DATA_MODE                   (N_PROF) object 'D' 'D' 'D' ... 'D' 'D' 'D'
    TEMP_QC                          (N_PROF, N_LEVELS) int64 1 1 ... 99999
    TIME_QC                          (N_PROF) int64 1 1 1 1 1 1 ... 1 1 1 1 1 1
Attributes: (8)

In [25]:
# temp = example_bgc.to_dataframe().reset_index()
# temp[temp.POSITION_QC == 0]

,N_PROF,N_LEVELS,BBP700,BBP700_ADJUSTED,BBP700_ADJUSTED_ERROR,BBP700_ADJUSTED_QC,BBP700_DATA_MODE,BBP700_QC,CHLA,CHLA_ADJUSTED,...,TEMP,TEMP_ADJUSTED,TEMP_ADJUSTED_ERROR,TEMP_ADJUSTED_QC,TEMP_DATA_MODE,TEMP_QC,TIME_QC,LATITUDE,LONGITUDE,TIME
0,0,0,NaN,NaN,NaN,0,A,0,NaN,NaN,...,3.634,3.634,0.002,1,D,1,1,-51.355,30.208,2019-05-01 08:58:59.000999936
1,0,1,NaN,NaN,NaN,0,A,0,NaN,NaN,...,3.634,3.634,0.002,1,D,1,1,-51.355,30.208,2019-05-01 08:58:59.000999936
2,0,2,NaN,NaN,NaN,0,A,0,NaN,NaN,...,3.634,3.634,0.002,1,D,1,1,-51.355,30.208,2019-05-01 08:58:59.000999936
3,0,3,NaN,NaN,NaN,0,A,0,NaN,NaN,...,3.634,3.634,0.002,1,D,1,1,-51.355,30.208,2019-05-01 08:58:59.000999936
5,0,5,NaN,NaN,NaN,0,A,0,NaN,NaN,...,3.634,3.634,0.002,1,D,1,1,-51.355,30.208,2019-05-01 08:58:59.000999936
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92774,165,539,NaN,NaN,NaN,0,A,0,NaN,NaN,...,3.142,3.142,0.002,1,D,1,1,-47.723,111.012,2023-10-13 19:34:23.002000128
92775,165,540,NaN,NaN,NaN,0,A,0,NaN,NaN,...,3.139,3.139,0.002,1,D,1,1,-47.723,111.012,2023-10-13 19:34:23.002000128
92776,165,541,NaN,NaN,NaN,0,A,0,NaN,NaN,...,3.130,3.130,0.002,1,D,1,1,-47.723,111.012,2023-10-13 19:34:23.002000128
92777,165,542,NaN,NaN,NaN,0,A,0,NaN,NaN,...,3.125,3.125,0.002,1,D,1,1,-47.723,111.012,2023-10-13 19:34:23.002000128


In [13]:
import gsw
import mod_cremas as crx 
import mod_ocean as myocn
from mod_ocean import ytd2datetime, datetime2ytd

# Custom module
import mod_argo 
reload(mod_argo)

<module 'mod_argo' from '/Users/sangminsong/Library/CloudStorage/OneDrive-UW/Code/CREMAS/src/mod_argo.py'>

In [ ]:
# Example from SOGOS. For BGC
#  def make_float_df(wmo, floatDSdict):
#         """
#         Return dataframe from a single float dataset
#         @param:
#                 wmo (int)
#                 floatDSdict (dict): dictionary of float datasets
#         """
#         temp = floatDSdict[str(wmo)]
#         float_df = temp[['JULD','LATITUDE', 'LONGITUDE','PRES_ADJUSTED','TEMP_ADJUSTED','PSAL_ADJUSTED',
#                          'DOXY_ADJUSTED','NITRATE_ADJUSTED', # , 'PH_IN_SITU_TOTAL_ADJUSTED','BBP700_ADJUSTED', 
#                         'JULD_QC', 'POSITION_QC', 'PRES_ADJUSTED_QC', 'TEMP_ADJUSTED_QC','PSAL_ADJUSTED_QC', 
#                         'DOXY_ADJUSTED_QC', 'NITRATE_ADJUSTED_QC']].to_dataframe() #'BBP700_ADJUSTED_QC' #, 'PH_IN_SITU_TOTAL_ADJUSTED_QC',
#         dtimes = pd.to_datetime(float_df.JULD.values, unit='D', origin=pd.Timestamp('1950-01-01'))
#         float_df['yearday'] = sg.datetime2ytd(dtimes)
#         float_df['wmoid'] = np.repeat(wmo, len(float_df))


#         # need to get the profile number from the index
#         # create a 10-digit unique id so easy to sort later
#         prof = float_df.index.get_level_values(0)
#         prof = prof.astype(str); prof = [tag.zfill(3) for tag in prof]
#         float_df['profid'] = [str(wmo)+tag for tag in prof]


#         qc_keys = ['JULD_QC', 'POSITION_QC', 'PRES_ADJUSTED_QC', 'TEMP_ADJUSTED_QC', 'PSAL_ADJUSTED_QC',
#             'DOXY_ADJUSTED_QC', 'NITRATE_ADJUSTED_QC'] #, 'PH_IN_SITU_TOTAL_ADJUSTED_QC', 'BBP700_ADJUSTED_QC']
#         for key in qc_keys:  #qc flags are not stored as ints so we can convert
#                 newlist = []
#                 for qc in float_df[key]:
#                         if str(qc)[2] == 'n': newlist.append('NaN')
#                         else: newlist.append(str(qc)[2])
#                 float_df[key] = newlist
        
#         float_df.rename(columns={'LATITUDE':'lat','LONGITUDE':'lon',
#                         'PRES_ADJUSTED': 'pressure', 'TEMP_ADJUSTED': 'temperature', 'PSAL_ADJUSTED': 'salinity',
#                         'DOXY_ADJUSTED': 'oxygen', 'NITRATE_ADJUSTED':'nitrate'}, inplace=True)
#                         # 'PH_IN_SITU_TOTAL_ADJUSTED': 'pH', 'BBP700_ADJUSTED': 'bbp700'
#         float_df.rename(columns={'JULD_QC': 'juld_qc', 'POSITION_QC': 'position_qc', 
#                         'PRES_ADJUSTED_QC': 'pressure_qc', 'TEMP_ADJUSTED_QC': 'temperature_qc','PSAL_ADJUSTED_QC': 'salinity_qc',
#                         'DOXY_ADJUSTED_QC': 'oxygen_qc', 'NITRATE_ADJUSTED_QC': 'nitrate_qc'}, inplace=True) 
#                         # 'PH_IN_SITU_TOTAL_ADJUSTED_QC': 'pH_qc','BBP700_ADJUSTED_QC': 'bbp700_qc'

        
#         float_df['SA']= gsw.SA_from_SP(float_df['salinity'],float_df['pressure'],float_df['lon'],float_df['lat'])
#         float_df['CT'] = gsw.CT_from_t(float_df['SA'], float_df['temperature'], float_df['pressure']) # change Sp to SA, jun24

#         # # Add training variables Sigma0 and Spice as desired
#         float_df['sigma0'] = gsw.sigma0(float_df.SA.values, float_df.CT.values)
#         float_df['spice'] = gsw.spiciness0(float_df["SA"].values, float_df["CT"].values)

#         # Add oxygen saturation and drho/dz
#         float_df = float_df[dvars + qcvars]

#         return float_df


In [34]:
floatDF = example_bgc.to_dataframe().reset_index()

In [76]:
# Modify to work for bgc-floats, jan 29
# this is a modified version of mod_argo function, process_core_float()

def process_argo_float(float_df, bgc_list = ['pH'], ref_time = '2014-01-01'):
    """
    Return dataframe from a single float dataset accessed with Argopy. 
    Assumed to be used in 'expert' mode, i.e. not quality-controlled yet for BGC. 

    @param:
            float_df (dict):  single float Dataframe, accessed in 'research' mode for core
                                only 'expert' mode available for BGC floats
            bgc_list : list of BGC variables to include in the final dataframe
                        default ['pH'] adds only pH data
            ref_time : reference time for yearday calculation
    @return:
            float_df (pd.DataFrame): 
    """
    float_df = float_df.reset_index()


    # Default columns to rename, starting with necessary properties across core/bgc
    # Note that Argopy "research mode" has removed "ADJUSTED" from column names
    new_columns = {'LATITUDE':'latitude','LONGITUDE':'longitude', 'TIME':'datetime', 
                'CYCLE_NUMBER':'cycle_number', 'PLATFORM_NUMBER':'wmoid', 
                'PRES_ADJUSTED':'pressure', 'TEMP_ADJUSTED':'temperature', 'PSAL_ADJUSTED':'salinity'}
    # Rename QC and error columns
    new_columns.update({'TIME_QC': 'time_qc', 'POSITION_QC': 'position_qc', 
                        'PRES_ADJUSTED_QC': 'pressure_qc', 
                        'TEMP_ADJUSTED_QC': 'temperature_qc','PSAL_ADJUSTED_QC': 'salinity_qc'})
    new_columns.update({'PRES_ADJUSTED_ERROR': 'pres_error', 
                        'PSAL_ADJUSTED_ERROR': 'psal_error', 'TEMP_ADJUSTED_ERROR': 'temp_error'})
    
    # output_vars = new_columns.values()


    # if bgc_list == 'phys': # research mode
    #     new_columns = {'LATITUDE':'latitude','LONGITUDE':'longitude', 'TIME':'datetime', 'CYCLE_NUMBER':'cycle_number',
    #                             'PLATFORM_NUMBER':'wmoid', 'PRES':'pressure', 'TEMP':'temperature', 'PSAL':'salinity',
    #                             'PRES_ERROR': 'pres_error', 'PSAL_ERROR': 'psal_error', 'TEMP_ERROR': 'temp_error'}
    
    if 'pH' in bgc_list: # expert mode
        new_columns.update({'PH_IN_SITU_TOTAL_ADJUSTED': 'pH', 'PH_IN_SITU_TOTAL_ADJUSTED_QC': 'pH_qc',
                            'PH_IN_SITU_TOTAL_ADJUSTED_ERROR': 'pH_error'})
    if 'oxygen' in bgc_list: 
        new_columns.update({'DOXY_ADJUSTED': 'oxygen', 'DOXY_ADJUSTED_QC': 'oxygen_qc',
                            'DOXY_ADJUSTED_ERROR': 'oxygen_error'})
    # if 'nitrate' in bgc_list:
    #     new_columns.update({'NITRATE_ADJUSTED': 'nitrate', 'NITRATE_ADJUSTED_QC': 'nitrate_qc',
    #                         'NITRATE_ADJUSTED_ERROR': 'nitrate_error'})
        
    float_df.rename(columns=new_columns, inplace=True)
    float_df['yearday'] = myocn.datetime2ytd(float_df['datetime'], ref_time = ref_time)

    # Create a unique profile id to be a useful index
    # Make sure strings are filled so 1st and 10th profile are different
    prof = [tag.zfill(3) for tag in float_df['cycle_number'].astype(str)]
    float_df['profid'] = [str(float_df.wmoid[0]) + '_id' + tag for tag in prof]

    # Add calculated variables using gsw
    float_df['SA']= gsw.SA_from_SP(float_df['salinity'],float_df['pressure'],float_df['longitude'],float_df['latitude'])
    float_df['CT'] = gsw.CT_from_t(float_df['SA'], float_df['temperature'], float_df['pressure']) 
    float_df['sigma0'] = gsw.sigma0(float_df.SA.values, float_df.CT.values)
    float_df['spice'] = gsw.spiciness0(float_df["SA"].values, float_df["CT"].values)

    # Standard variable list to return (core)
    # Reorder 
    output_vars = ['wmoid', 'profid', 'latitude', 'longitude', 'datetime', 'yearday',
            'pressure', 'CT', 'SA', 'sigma0', 'spice',
            'temperature', 'salinity',
            'temperature_qc', 'salinity_qc', 'pressure_qc',
            'time_qc', 'position_qc',
            'temp_error', 'psal_error', 'pres_error']
    
    for x in bgc_list:
        output_vars = output_vars + [x, x+'_qc', x+'_error']
    
    # if 'pH' in bgc_list:
    #     output_vars = output_vars + ['pH', 'pH_qc', 'pH_error']

    # Turn all QC flags into strings
    qc_vars = [var for var in float_df.columns.tolist() if '_qc' in var]
    for k in qc_vars:
        float_df[k] = float_df[k].astype(str)

    print(output_vars)
    
    return float_df[output_vars]




In [83]:
trial = process_argo_float(floatDF, bgc_list = ['pH'], ref_time = '2014-01-01')

['wmoid', 'profid', 'latitude', 'longitude', 'datetime', 'yearday', 'pressure', 'CT', 'SA', 'sigma0', 'spice', 'temperature', 'salinity', 'temperature_qc', 'salinity_qc', 'pressure_qc', 'time_qc', 'position_qc', 'temp_error', 'psal_error', 'pres_error', 'pH', 'pH_qc', 'pH_error']


In [84]:
qc_vars = [var for var in trial.columns.tolist() if '_qc' in var]
qc_vars

['temperature_qc',
 'salinity_qc',
 'pressure_qc',
 'time_qc',
 'position_qc',
 'pH_qc']

In [ ]:
['temperature_qc', 'salinity_qc', 'pressure_qc', 'time_qc', 'position_qc', 'pH_qc']

In [101]:
def filter_qc_flags(float_df, qc_vars = 'all', use_flags=['1', '2', '5', '8']):
        """
        Filter a dataframe based on QC flags.
        Can choose different QC flags for different variables by calling the function multiple times.
        @param: float_df (pd.DataFrame): dataframe of float data
                qc_vars (list): list of QC variables to filter
                        default 'all' filters on any variable with '_qc' in the name
                use_flags : flags that pass QC; default are standard argo QC flags 1, 2, and 8
                        '1' for 'good' data (only '1' returned in 'research' mode)
                        '2' for 'probably good' data
                        '5' for 'changed' data (rare; for position qc where lat/lon was adjusted)
                        '8' for 'interpolated/estimated' data
        @return: float_qc (pd.DataFrame)
        """ 
        print('Using flags: ', use_flags)
        float_qc = float_df.copy()
        print('# of obs before QC filtering: \t\t', len(float_qc))

        if qc_vars == 'all':
                qc_vars = [var for var in float_qc.columns.tolist() if '_qc' in var]
        
        for var in qc_vars:
                float_qc = float_qc[float_qc[var].isin(use_flags)]
                print('# of obs after ', var, ': \t\t', len(float_qc))
                
        return float_qc
        

trial_qc_phys = filter_qc_flags(trial, qc_vars = ['temperature_qc', 'salinity_qc', 'pressure_qc', 'time_qc', 'position_qc'], use_flags=['1', '2', '5', '8'])
# trial_qc_all = filter_qc_flags(trial_qc_phys, qc_vars = ['pH_qc'], use_flags=['1', '2'])

Using flags:  ['1', '2', '5', '8']
# of obs before QC filtering: 		 92794
# of obs after  temperature_qc : 		 92111
# of obs after  salinity_qc : 		 91856
# of obs after  pressure_qc : 		 91856
# of obs after  time_qc : 		 91856
# of obs after  position_qc : 		 91856


In [ ]:
# Make dataframes using the custom Argo module
# reload(mod_argo)
# temp = core_profiles.to_dataframe()
# core_dict = {key:mod_argo.process_core_float(group, var_list = 'default', 
#                                     ref_time= '2014-01-01') for key,group in temp.groupby('PLATFORM_NUMBER')}
